In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# 이상치 전후 시계열 매핑을 위해 행/열 축소옵션 취소
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'data' / '1_train_full_fe.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '1_test_fe.parquet')

print(train.shape, test.shape)

(749657, 34) (186263, 33)


In [2]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = train.select_dtypes(include='number').columns

assert len(num_cols) == 32, "'num_cols' 개수 이상"
assert (set(train.columns) - set(num_cols)) == {'station_id', 'timestamp'}, "'num_cols' 확인필요"

print(num_cols)  # target_col 포함됨

Index(['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h',
       'wl_t_minus_12h', 'wl_t_minus_24h', 'imerg_rain_1h', 'imerg_rain_6h',
       'imerg_rain_24h', 'sfc_rain_1h', 'sfc_rain_6h', 'sfc_rain_24h',
       'radar_reflectivity_mean', 'radar_reflectivity_p95',
       'radar_reflectivity_ge20_fraction', 'cat_area', 'wsarea',
       'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio',
       'stream_order', 'target_maxrise_6h', 'wl_diff_1h', 'wl_diff_3h',
       'wl_diff_6h', 'wl_diff_12h', 'wl_diff_24h', 'imerg_int_1to6h',
       'imerg_int_6to24h', 'sfc_int_1to6h', 'sfc_int_6to24h'],
      dtype='object')


## 2. 이상치 제거

In [3]:
# ==============================================================
# 2-1. 이상치(계측오류) 제거: train 종속변수 이상치 제거(1개)
# ==============================================================
# 변수별 분포특성 확인단계에서 확인됨

# 이상치 탐색
print('==================== 이상치 탐색 ====================')
display(
    train.loc[train[target_col] < -2.5][
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()

# 이상치 전후 시계열 확인
cond_maxrise_out_st1 = train['station_id'] == 'ST0007'
cond_maxrise_out_time = train['timestamp'].between(
    '2103-02-27 03:00', '2103-02-28 00:00'
)
print('==================== 이상치 전후 시계열 ====================')
display(
    train.loc[
        (cond_maxrise_out_st1 & cond_maxrise_out_time),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()

# 이상치 조건설정: 전후 시계열에서 종속변수 -8.74는 이상치로 판단
cond_maxrise_out = (
    cond_maxrise_out_st1 &
    cond_maxrise_out_time &
    (train['target_maxrise_6h'] < -8)
)
                
print(f"이상치 개수(제거대상): {cond_maxrise_out.sum()}")
print(f"이상치 행 제거 전 데이터 개수: {len(train)}") 
assert cond_maxrise_out.sum() == 1, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train = train.drop(
    index = train.loc[cond_maxrise_out].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train)}")

==================== 이상치 탐색 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000583385,2103-02-27 05:00:00,ST0007,8.75,7.4,4.53,2.31,2.37,2.5,-8.74



==================== 이상치 전후 시계열 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000128421,2103-02-27 03:00:00,ST0007,5.35,4.53,2.41,2.31,2.40,2.49,3.40
TR_000397383,2103-02-27 04:00:00,ST0007,7.40,5.35,2.96,2.31,2.38,2.49,1.35
TR_000583385,2103-02-27 05:00:00,ST0007,8.75,7.40,4.53,2.31,2.37,2.50,-8.74



이상치 개수(제거대상): 1
이상치 행 제거 전 데이터 개수: 749657
이상치 행 제거 후 train셋 행 개수: 749656


In [4]:
# ======================================================================
# 2-2. 이상치(계측오류) 제거: train 수위변수 이상치 제거(ST0445, 12개)
# ======================================================================
# 변수별 분포특성 확인단계에서 확인됨
# 수위차분변수 산점도에서 이상치로 확인

# 이상치 탐색
cond_out1 = (train[wl_diff_cols] < -10).any(axis=1)  # axis=1: 행별 검사

print('==================== 이상치 탐색 ====================')
display(
    train.loc[cond_out1][
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
    .sort_values(by=['station_id', 'timestamp'])
)
print()

# 이상치 전후 시계열 확인
cond_wl_out_st1 = train['station_id'] == 'ST0445'
cond_wl_out_time1 = (
    train['timestamp'].between('2100-08-09 00:00:00', '2100-08-10 00:00:00') |
    train['timestamp'].between('2102-09-17 00:00:00', '2102-09-19 00:00:00') |
    train['timestamp'].between('2101-11-02 00:00:00', '2101-11-03 00:00:00')
)
print('==================== 이상치 전후 시계열 ====================')
display(
    train.loc[
        (cond_wl_out_st1 & cond_wl_out_time1),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()

# 이상치 조건설정: 전후 시계열에서 ST0445 관측소는의 수위 17.63은 비정상적 상승으로 이상치로 판단
cond_wl_out1 = (
    cond_wl_out_st1 &
    cond_wl_out_time1 &
    ((train[wl_cols] > 17).any(axis=1))
)
                
print(f"이상치 개수(제거대상): {cond_wl_out1.sum()}")
print(f"이상치 행 제거 전 데이터 개수: {len(train)}") 
assert cond_wl_out1.sum() == 12, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train = train.drop(
    index = train.loc[cond_wl_out1].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train)}")

==================== 이상치 탐색 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000437549,2102-04-01 00:00:00,ST0066,11.60,11.60,11.55,11.47,11.34,23.45,0.15
TR_000386997,2100-08-09 15:00:00,ST0445,2.61,2.61,17.63,17.63,17.63,17.63,0.00
TR_000202940,2101-11-02 07:00:00,ST0445,2.58,2.58,2.58,2.59,17.63,17.63,0.01
TR_000205403,2101-11-02 09:00:00,ST0445,2.58,2.58,2.58,2.59,17.63,17.63,0.01
TR_000589829,2102-09-18 06:00:00,ST0445,2.61,17.63,17.63,17.63,17.63,17.63,0.00
TR_000202521,2102-09-18 07:00:00,ST0445,2.61,2.61,17.63,17.63,17.63,17.63,0.00
TR_000652873,2102-09-18 08:00:00,ST0445,2.61,2.61,17.63,17.63,17.63,17.63,0.00
TR_000182368,2102-09-18 09:00:00,ST0445,2.59,2.61,2.61,17.63,17.63,17.63,0.02
TR_000333157,2102-09-18 10:00:00,ST0445,2.61,2.59,2.61,17.63,17.63,17.63,0.00



==================== 이상치 전후 시계열 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000386997,2100-08-09 15:00:00,ST0445,2.61,2.61,17.63,17.63,17.63,17.63,0.00
TR_000202940,2101-11-02 07:00:00,ST0445,2.58,2.58,2.58,2.59,17.63,17.63,0.01
TR_000205403,2101-11-02 09:00:00,ST0445,2.58,2.58,2.58,2.59,17.63,17.63,0.01
TR_000639479,2102-09-18 01:00:00,ST0445,17.63,17.63,17.63,17.63,17.63,17.63,0.00
TR_000561875,2102-09-18 03:00:00,ST0445,17.63,17.63,17.63,17.63,17.63,17.63,0.00
TR_000258437,2102-09-18 04:00:00,ST0445,17.63,17.63,17.63,17.63,17.63,17.63,0.00
TR_000589829,2102-09-18 06:00:00,ST0445,2.61,17.63,17.63,17.63,17.63,17.63,0.00
TR_000202521,2102-09-18 07:00:00,ST0445,2.61,2.61,17.63,17.63,17.63,17.63,0.00
TR_000652873,2102-09-18 08:00:00,ST0445,2.61,2.61,17.63,17.63,17.63,17.63,0.00



이상치 개수(제거대상): 12
이상치 행 제거 전 데이터 개수: 749656
이상치 행 제거 후 train셋 행 개수: 749644


In [5]:
# ======================================================================
# 2-3. 이상치(계측오류) 제거: train 수위변수 이상치 제거(ST0066, 20개)
# ======================================================================
# 변수별 분포특성 확인단계에서 확인됨

# 이상치 관측소: 2-2에서 확인됨
cond_wl_out_st2 = train['station_id'] == 'ST0066'

# ST0066 관측소의 현재 수위(wl_t) 분위수
# 0.99분위수가 23.55, 0.98분위수가 12.45 -> 23 이상은 계측오류로 판단
print('=============== ST0066관측소 현재수위 분위수 ===============')
display(
    train.loc[
        cond_wl_out_st2, 'wl_t'
    ].quantile([0.5, 0.9, 0.95, 0.98, 0.99, 1.00])
)
print()

# 이상치 전후 시계열 확인
cond_wl_out_time2 = (
    train['timestamp'].between('2100-08-19 15:00:00', '2100-08-20 00:00:00') |
    train['timestamp'].between('2102-03-31 00:00:00', '2102-04-01 03:00:00')
)

print('==================== 이상치 전후 시계열 ====================')
display(
    train.loc[
        (cond_wl_out_st2 & cond_wl_out_time2),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()

# 이상치 조건설정
cond_wl_out2 = (
    cond_wl_out_st2 &
    ((train[wl_cols] > 23).any(axis=1))
)
                
print(f"이상치 개수(제거대상): {cond_wl_out2.sum()}")
print(f"이상치 행 제거 전 데이터 개수: {len(train)}") 
assert cond_wl_out2.sum() == 20, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train = train.drop(
    index = train.loc[cond_wl_out2].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train)}")

=============== ST0066관측소 현재수위 분위수 ===============


0.50     6.580
0.90    10.724
0.95    11.010
0.98    12.450
0.99    23.550
1.00    23.560
Name: wl_t, dtype: float64


==================== 이상치 전후 시계열 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000616893,2100-08-19 15:00:00,ST0066,23.55,23.55,23.55,23.56,23.56,23.54,0.01
TR_000463562,2100-08-19 16:00:00,ST0066,23.55,23.55,23.55,23.56,23.56,23.55,0.02
TR_000677096,2100-08-19 17:00:00,ST0066,23.55,23.55,23.55,23.53,23.56,23.55,0.02
TR_000446779,2100-08-19 18:00:00,ST0066,23.55,23.55,23.55,23.55,23.56,23.55,0.02
TR_000437549,2102-04-01 00:00:00,ST0066,11.60,11.60,11.55,11.47,11.34,23.45,0.15
TR_000252105,2102-04-01 01:00:00,ST0066,11.63,11.60,11.57,11.48,11.36,11.08,0.17
TR_000598157,2102-04-01 02:00:00,ST0066,11.65,11.63,11.60,11.53,11.39,11.11,0.15
TR_000496451,2102-04-01 03:00:00,ST0066,11.66,11.65,11.60,11.55,11.41,11.14,0.19



이상치 개수(제거대상): 20
이상치 행 제거 전 데이터 개수: 749644
이상치 행 제거 후 train셋 행 개수: 749624


In [6]:
# 이상치 탐색
cond_out2 = (train['wl_diff_12h'] < -4)  # axis=1: 행별 검사

print('==================== 이상치 탐색 ====================')
display(
    train.loc[cond_out2][
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
    .sort_values(by=['station_id', 'timestamp'])
)
print()


==================== 이상치 탐색 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000547855,2102-01-01 07:00:00,ST0089,0.03,0.01,0.01,6.38,6.38,6.38,0.09
TR_000457689,2102-01-01 10:00:00,ST0089,0.11,0.10,0.03,0.01,6.38,6.38,0.06
TR_000232755,2103-06-06 16:00:00,ST0204,3.25,3.41,3.87,4.97,7.32,1.02,0.66
TR_000709250,2103-06-07 20:00:00,ST0221,7.56,7.82,8.44,9.62,11.81,8.52,-0.22
TR_000258655,2103-06-07 21:00:00,ST0221,7.34,7.56,8.12,9.19,11.66,8.49,-0.20
TR_000168260,2103-06-07 22:00:00,ST0221,7.14,7.34,7.82,8.79,11.39,8.44,-0.20
TR_000447715,2103-06-07 23:00:00,ST0221,6.94,7.14,7.56,8.44,11.00,8.32,-0.17
TR_000505008,2103-06-09 13:00:00,ST0412,6.89,7.34,8.35,9.81,10.92,0.35,-0.42
TR_000602451,2103-06-09 14:00:00,ST0412,6.47,6.89,7.83,9.39,11.37,0.72,-0.36


In [7]:
# 이상치 탐색
cond_out3 = ~(train['wl_diff_24h'].between(-6.5, 5))  # axis=1: 행별 검사

print('==================== 이상치 탐색 ====================')
display(
    train.loc[cond_out3][
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
    .sort_values(by=['station_id', 'timestamp'])
)
print()

==================== 이상치 탐색 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000715299,2103-12-02 00:00:00,ST0007,0.00,0.00,0.00,0.00,0.00,8.75,0.00
TR_000175145,2101-03-08 21:00:00,ST0024,18.84,18.22,17.15,15.80,13.95,13.53,1.59
TR_000304012,2101-03-08 20:00:00,ST0033,6.85,6.72,4.80,2.08,1.14,1.12,-0.23
TR_000034524,2101-03-08 21:00:00,ST0033,6.62,6.85,6.00,3.17,1.14,1.12,-0.37
TR_000264425,2103-02-06 20:00:00,ST0066,5.15,4.16,0.91,0.36,0.10,0.04,2.67
TR_000636924,2103-02-06 21:00:00,ST0066,6.02,5.15,2.98,0.38,0.13,0.04,1.86
TR_000560608,2103-02-06 22:00:00,ST0066,6.75,6.02,4.16,0.39,0.13,0.04,1.20
TR_000179263,2103-02-06 23:00:00,ST0066,7.25,6.75,5.15,0.91,0.26,0.04,0.77
TR_000604908,2101-03-08 17:00:00,ST0071,5.96,5.39,4.00,2.02,0.90,0.91,2.83


In [8]:
# ==========================================================================
# 2-4. 이상치(계측오류) 제거: train 수위변수 이상치 제거(4개 관측소, 15개)
# ==========================================================================
# train, test셋 분포 비교에서 확인
# 전후 시계열을 확인하여 계측오류 임계치를 설정, 이상치 제거

# ST0089관측소 이상치 전후 시계열 확인
cond_wl_out_st3 = train['station_id'] == 'ST0089'
cond_wl_out_time3 = (
    train['timestamp'].between('2102-01-01 00:00', '2102-01-02 00:00')
)
print('==================== ST0089관측소 이상치 전후 시계열 ====================')
display(
    train.loc[
        (cond_wl_out_st3 & cond_wl_out_time3),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0089관측소 이상치 조건 설정
cond_wl_out3 = (
    cond_wl_out_st3 &
    cond_wl_out_time3 &
    ((train[wl_cols] > 6).any(axis=1))
)


# ST0158관측소 이상치 전후 시계열 확인
cond_wl_out_st4 = train['station_id'] == 'ST0158'
cond_wl_out_time4 = (
    train['timestamp'].between('2102-05-10 0:00', '2102-05-12 00:00')
)
print('==================== ST0158관측소 이상치 전후 시계열 ====================')
display(
    train.loc[
        (cond_wl_out_st4 & cond_wl_out_time4),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0158관측소 이상치 조건 설정
cond_wl_out4 = (
    cond_wl_out_st4 &
    cond_wl_out_time4 &
    ((train[wl_cols] > 7).any(axis=1))
)


# ST0007관측소 이상치 전후 시계열 확인
cond_wl_out_st5 = train['station_id'] == 'ST0007'
cond_wl_out_time5 = (
    train['timestamp'].between('2103-12-01 00:00', '2103-12-02 02:00')
)
print('==================== ST0007관측소 이상치 전후 시계열 ====================')
display(
    train.loc[
        (cond_wl_out_st5 & cond_wl_out_time5),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0007관측소 이상치 조건 설정
cond_wl_out5 = (
    cond_wl_out_st5 &
    cond_wl_out_time5 &
    ((train[wl_cols] > 8).any(axis=1))
)


# ST0448관측소 이상치 전후 시계열 확인
cond_wl_out_st6 = train['station_id'] == 'ST0448'
cond_wl_out_time6 = (
    train['timestamp'].between('2103-12-02 00:00', '2103-12-03 00:00')
)
print('==================== ST0448관측소 이상치 전후 시계열 ====================')
display(
    train.loc[
        (cond_wl_out_st6 & cond_wl_out_time6),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0448관측소 이상치 조건 설정
cond_wl_out6 = (
    cond_wl_out_st6 &
    cond_wl_out_time6 &
    ((train[wl_cols] > 9).any(axis=1))
)

                
print(
    f"이상치 개수(제거대상): {
        (cond_wl_out3 | cond_wl_out4 | cond_wl_out5 | cond_wl_out6)
        .sum()
    }"
)
print(f"이상치 행 제거 전 데이터 개수: {len(train)}") 
assert (cond_wl_out3 | cond_wl_out4 | cond_wl_out5 | cond_wl_out6).sum() == 15, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train = train.drop(
    index = train.loc[
        (cond_wl_out3 | cond_wl_out4 | cond_wl_out5 | cond_wl_out6)
    ].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train)}")

==================== ST0089관측소 이상치 전후 시계열 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000547855,2102-01-01 07:00:00,ST0089,0.03,0.01,0.01,6.38,6.38,6.38,0.09
TR_000457689,2102-01-01 10:00:00,ST0089,0.11,0.10,0.03,0.01,6.38,6.38,0.06
TR_000318564,2102-01-01 21:00:00,ST0089,0.47,0.42,0.18,0.16,0.10,6.38,0.07



==================== ST0158관측소 이상치 전후 시계열 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000193482,2102-05-11 00:00:00,ST0158,0.0,0.0,0.0,0.0,0.0,7.39,0.0
TR_000057205,2102-05-11 01:00:00,ST0158,0.0,0.0,0.0,0.0,0.0,7.39,0.0
TR_000154692,2102-05-11 02:00:00,ST0158,0.0,0.0,0.0,0.0,0.0,7.39,0.0



==================== ST0007관측소 이상치 전후 시계열 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000715299,2103-12-02 00:00:00,ST0007,0.0,0.0,0.0,0.0,0.0,8.75,0.0
TR_000581814,2103-12-02 01:00:00,ST0007,0.0,0.0,0.0,0.0,0.0,0.01,0.0
TR_000127840,2103-12-02 02:00:00,ST0007,0.0,0.0,0.0,0.0,0.0,0.00,0.0



==================== ST0448관측소 이상치 전후 시계열 ====================


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,,
TR_000468355,2103-12-02 00:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.59,0.0
TR_000450715,2103-12-02 01:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.59,0.0
TR_000267036,2103-12-02 02:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.59,0.0
TR_000476408,2103-12-02 03:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.57,0.0
TR_000185855,2103-12-02 04:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.59,0.0
TR_000385460,2103-12-02 05:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.59,0.0
TR_000665570,2103-12-02 06:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.59,0.0
TR_000480061,2103-12-02 07:00:00,ST0448,0.0,0.0,0.0,0.0,0.0,9.59,0.0



이상치 개수(제거대상): 15
이상치 행 제거 전 데이터 개수: 749624
이상치 행 제거 후 train셋 행 개수: 749609


In [9]:
# ============================================================
# 3-4. 최종 train, test 저장
# ============================================================

train.to_parquet(PROJECT_ROOT / 'data' / '2_train_full_clean.parquet')
test.to_parquet(PROJECT_ROOT / 'data' / '2_test_clean.parquet')

print("Parquet 저장완료")

Parquet 저장완료
